# Урок 16. Anomaly Detection — поиск аномалий
**Библиотеки:** numpy, matplotlib, sklearn

**Цель:** находить «странные» примеры: фрод, брак, сбои.

## Простыми словами
Аномалия — точка, непохожая на остальные. Идея простейшего детектора:
выучить, что такое «нормально» (среднее и разброс), и помечать всё, что слишком далеко.

**Гауссовский подход (как у Andrew Ng):** считаем, что нормальные данные — колокол.
Для новой точки считаем вероятность по колоколу: вероятность крошечная → аномалия.
Практическое правило: дальше 3 стандартных отклонений от среднего → подозрительно.

Где применяется: банковский антифрод, мониторинг серверов, контроль качества на заводе,
медицинские отклонения. Почему не классификация? Аномалий слишком мало и они всё время новые.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Транзакции: суммы покупок клиента + несколько фродовых
rng = np.random.default_rng(11)
normal = rng.normal(50, 15, 300)                # обычные покупки ~50$
fraud = np.array([180, 220, 5, 250, 199])        # странные
amounts = np.concatenate([normal, fraud])

mu, sigma = amounts.mean(), amounts.std()        # учим "нормальность"
z = np.abs(amounts - mu) / sigma                 # на сколько сигм точка отклоняется
is_anomaly = z > 3                                # правило трёх сигм

print(f"mu = {mu:.1f}, sigma = {sigma:.1f}")
print("Найдено аномалий:", is_anomaly.sum())
print("Суммы аномалий:", np.sort(amounts[is_anomaly]).round(0))

plt.hist(amounts[~is_anomaly], bins=30, label='норма')
plt.scatter(amounts[is_anomaly], np.zeros(is_anomaly.sum()), c='red', s=80, zorder=3, label='аномалии')
plt.legend(); plt.grid(True); plt.title('Детекция по правилу 3 сигм'); plt.show()

In [ ]:
# Готовый инструмент для многомерных данных: IsolationForest
from sklearn.ensemble import IsolationForest
X = np.column_stack([amounts, rng.normal(12, 3, len(amounts))])   # сумма + час покупки
iso = IsolationForest(contamination=0.02, random_state=0)  # ожидаем ~2% аномалий
labels = iso.fit_predict(X)                                 # -1 = аномалия, 1 = норма

plt.scatter(X[labels == 1, 0], X[labels == 1, 1], s=10, label='норма')
plt.scatter(X[labels == -1, 0], X[labels == -1, 1], c='red', s=50, label='аномалия')
plt.xlabel('сумма'); plt.ylabel('час'); plt.legend(); plt.grid(True); plt.show()

## Что произошло
Правило 3 сигм поймало почти все подложенные фродовые суммы без единого ответа y.
IsolationForest делает то же для многомерных данных: он «изолирует» точки случайными разрезами —
аномалии изолируются быстро, нормальные точки в гуще — медленно.

## Типичные ошибки
1. Учить mu и sigma на данных ВМЕСТЕ с аномалиями — они искажают «норму». В идеале учись на чистых.
2. Один порог для всех задач: 3 сигмы — стартовая точка, порог подбирают под цену ошибок.
3. Применять гауссовский метод к данным, которые совсем не «колокол» (например, два горба).

## Конспект 📓
Аномалия = далеко от нормы. Простой детектор: |x − mu| > 3*sigma.
IsolationForest(contamination=доля) для многомерного: -1 аномалия, 1 норма.
Не классификация, потому что аномалий мало и они новые каждый раз.

---
## Мини-задание 💪
Поменяй порог с 3 сигм на 2 и на 4. Сколько аномалий находится в каждом случае? Какой компромисс ты регулируешь этим порогом (вспомни precision/recall)?

## 5 проверочных вопросов ❓
1. Почему антифрод — это не обычная классификация?
2. В чём идея правила 3 сигм?
3. Что значит contamination=0.02 в IsolationForest?
4. Что вернёт IsolationForest для аномалии?
5. Когда гауссовский метод не сработает?

## Домашнее задание 🏠
Задача 44 из practice_tasks.md. Прогони IsolationForest на данных клиентов из урока 15 — кого он посчитал странным?
